# 01 · Understanding the NBA shot logs

This is the first EDA notebook. It explains what a row represents, what all 21 source columns mean, and which values deserve investigation before we build player-specific make probabilities.

The source is Dan Becker's 2014–15 NBA shot log. The archive covers part of the season and includes attempts for 281 shooters. Closest-defender distance is recorded on each row. The CSV lives locally in data/raw and the source audit documents provenance and publication status.

[Original dataset](https://www.kaggle.com/datasets/dansbecker/nba-shot-logs) · [Data source audit](../docs/data-sources.md)

## 1. Load the file and set the analysis window

The date is embedded in MATCHUP, so we parse it into a separate column. The full file is read to create the split. All previews and EDA below use games before February 1, 2015. Later games stay available for final evaluation.

In [1]:
from pathlib import Path

import pandas as pd

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent

path = root / "data" / "raw" / "shot_logs.csv"
if not path.is_file():
    raise FileNotFoundError(f"Place the extracted shot log at {path}")

holdout_start = pd.Timestamp("2015-02-01")
raw = pd.read_csv(path, dtype={"GAME_ID": "string"})
game_date = pd.to_datetime(
    raw["MATCHUP"].str.split(" - ", n=1).str[0],
    format="%b %d, %Y",
)
shots = raw.loc[game_date.lt(holdout_start)].assign(GAME_DATE=game_date)
source_columns = raw.columns.tolist()
print(f"Source file: {len(raw):,} rows and {len(source_columns)} columns")
print(f"EDA window: {len(shots):,} attempts in {shots['GAME_ID'].nunique():,} games")
print(f"EDA dates: {shots['GAME_DATE'].min():%Y-%m-%d} to {shots['GAME_DATE'].max():%Y-%m-%d}")

Source file: 128,069 rows and 21 columns
EDA window: 101,997 attempts in 710 games
EDA dates: 2014-10-28 to 2015-01-31


## 2. Understand a row

One row represents one recorded field goal attempt by one shooter. A game contains many attempts, and a shooter can appear in many games. GAME_ID identifies the game, player_id identifies the shooter, and SHOT_NUMBER is the sequence number for that shooter within the game. Together they identify the attempt in this archive.

The preview shows attempts from one shooter in one game so the relationship between those three fields is visible.

In [2]:
preview = shots.sort_values(["GAME_DATE", "GAME_ID", "player_id", "SHOT_NUMBER"])
first = preview.iloc[0]
same_shooter_game = preview["GAME_ID"].eq(first["GAME_ID"]) & preview["player_id"].eq(first["player_id"])
preview.loc[same_shooter_game, [
    "GAME_DATE", "GAME_ID", "player_name", "SHOT_NUMBER", "PERIOD",
    "GAME_CLOCK", "SHOT_DIST", "CLOSE_DEF_DIST", "SHOT_RESULT",
]].head()

,GAME_DATE,GAME_ID,player_name,SHOT_NUMBER,PERIOD,GAME_CLOCK,SHOT_DIST,CLOSE_DEF_DIST,SHOT_RESULT
44445,2014-10-28,21400001,ben gordon,1,1,3:11,20.9,3.6,missed
44446,2014-10-28,21400001,ben gordon,2,1,2:02,13.7,5.9,missed
44447,2014-10-28,21400001,ben gordon,3,1,0:29,22.1,2.9,missed
44448,2014-10-28,21400001,ben gordon,4,2,0:28,21.6,3.6,missed
44449,2014-10-28,21400001,ben gordon,5,3,3:29,14.6,3.0,made


## 3. Column guide

| Column | Meaning | Role in this project |
| --- | --- | --- |
| GAME_ID | Game identifier, kept as text | Group and match games |
| MATCHUP | Game date and team matchup from the shooter's perspective | Parse game date and identify the matchup |
| LOCATION | Whether the shooter's team played at home (H) or away (A) | Venue context |
| W | Whether the shooter's team won or lost | Postgame field |
| FINAL_MARGIN | Final scoring margin from the shooter's team's perspective | Postgame field |
| SHOT_NUMBER | Attempt sequence for this shooter in this game | Part of the shot key |
| PERIOD | Quarter number, with 5 onward indicating overtime | Game context |
| GAME_CLOCK | Minutes and seconds remaining in the period | Game context |
| SHOT_CLOCK | Seconds remaining on the possession clock | Shot context, with missing values |
| DRIBBLES | Number of dribbles before the attempt | Shot creation context |
| TOUCH_TIME | Seconds the shooter held the ball before the attempt | Shot creation context, with negative values to investigate |
| SHOT_DIST | Distance from the shooter to the basket in feet | Shot context |
| PTS_TYPE | Value of the attempt, 2 or 3 points | Shot context |
| SHOT_RESULT | Shot result as made or missed | Text version of the label |
| CLOSEST_DEFENDER | Name of the nearest defender | Descriptive identifier |
| CLOSEST_DEFENDER_PLAYER_ID | NBA ID of the nearest defender | Defender identifier |
| CLOSE_DEF_DIST | Recorded distance from the shooter to the nearest defender in feet | Essential model input |
| FGM | 1 for a made field goal and 0 for a miss | Binary target |
| PTS | Actual points scored on the attempt | Derived from the outcome |
| player_name | Shooter name | Display and lookup |
| player_id | NBA ID of the shooter | Build prior shooting history and group attempts |

GAME_DATE is added in this notebook. The two distance columns describe different things. For example, SHOT_DIST of 25 and CLOSE_DEF_DIST of 2 describe a shot from 25 feet with a defender 2 feet away. The archive has no court coordinates, so it does not identify the exact spot on the floor.

FGM is what we want to predict. SHOT_RESULT and PTS reveal that result. W and FINAL_MARGIN are known after the game. Leave these outcome and postgame fields out of the inputs.

In [3]:
pd.DataFrame({
    "dtype": shots[source_columns].dtypes.astype(str),
    "non_missing": shots[source_columns].count(),
    "missing": shots[source_columns].isna().sum(),
    "distinct_non_missing": shots[source_columns].nunique(),
})

,dtype,non_missing,missing,distinct_non_missing
GAME_ID,string,101997,0,710
MATCHUP,object,101997,0,1420
LOCATION,object,101997,0,2
W,object,101997,0,2
FINAL_MARGIN,int64,101997,0,84
SHOT_NUMBER,int64,101997,0,37
PERIOD,int64,101997,0,7
GAME_CLOCK,object,101997,0,719
SHOT_CLOCK,float64,97371,4626,241
DRIBBLES,int64,101997,0,33


In [4]:
shots[[
    "PERIOD", "SHOT_NUMBER", "PTS_TYPE", "SHOT_DIST", "CLOSE_DEF_DIST",
    "DRIBBLES", "TOUCH_TIME", "SHOT_CLOCK",
]].describe().T

,count,mean,std,min,25%,50%,75%,max
PERIOD,101997.0,2.473347,1.141894,1.0,1.0,2.0,3.0,7.0
SHOT_NUMBER,101997.0,6.508642,4.712822,1.0,3.0,5.0,9.0,37.0
PTS_TYPE,101997.0,2.264194,0.440905,2.0,2.0,2.0,3.0,3.0
SHOT_DIST,101997.0,13.564836,8.894046,0.0,4.7,13.6,22.5,47.2
CLOSE_DEF_DIST,101997.0,4.115065,2.750112,0.0,2.3,3.7,5.3,53.2
DRIBBLES,101997.0,2.023775,3.476678,0.0,0.0,1.0,2.0,32.0
TOUCH_TIME,101997.0,2.766764,3.054421,-163.6,0.9,1.6,3.7,24.9
SHOT_CLOCK,97371.0,12.438302,5.770275,0.0,8.1,12.3,16.6,24.0


## 4. Check unusual values

Start with structural checks, then count observations that deserve review. Zero defender distance might reflect rounding or a tracking artifact. Distances above 25 feet are flagged for inspection using a provisional threshold. Negative touch time is implausible for a duration. These checks leave every row intact.

In [5]:
shot_key = ["GAME_ID", "player_id", "SHOT_NUMBER"]
pd.Series({
    "Duplicate shot keys": shots.duplicated(shot_key).sum(),
    "Missing values in shot key": shots[shot_key].isna().any(axis=1).sum(),
    "Games with conflicting dates": shots.groupby("GAME_ID")["GAME_DATE"].nunique().gt(1).sum(),
    "Invalid or missing shot result": (~shots["SHOT_RESULT"].isin(["made", "missed"])).sum(),
    "FGM disagrees with SHOT_RESULT": shots["FGM"].ne(shots["SHOT_RESULT"].map({"made": 1, "missed": 0})).sum(),
    "PTS disagrees with FGM × PTS_TYPE": shots["PTS"].ne(shots["FGM"] * shots["PTS_TYPE"]).sum(),
    "Shot value other than 2 or 3": (~shots["PTS_TYPE"].isin([2, 3])).sum(),
}, name="count")

Duplicate shot keys                  0
Missing values in shot key           0
Games with conflicting dates         0
Invalid or missing shot result       0
FGM disagrees with SHOT_RESULT       0
PTS disagrees with FGM × PTS_TYPE    0
Shot value other than 2 or 3         0
Name: count, dtype: int64

In [6]:
clock_parts = shots["GAME_CLOCK"].str.split(":", expand=True).astype(int)
period_seconds = clock_parts[0] * 60 + clock_parts[1]

flags = pd.DataFrame({
    "Missing closest-defender distance": shots["CLOSE_DEF_DIST"].isna(),
    "Negative closest-defender distance": shots["CLOSE_DEF_DIST"].lt(0),
    "Zero closest-defender distance": shots["CLOSE_DEF_DIST"].eq(0),
    "Closest-defender distance above 25 ft": shots["CLOSE_DEF_DIST"].gt(25),
    "Zero distance to basket": shots["SHOT_DIST"].eq(0),
    "Negative touch time": shots["TOUCH_TIME"].lt(0),
    "Missing shot clock": shots["SHOT_CLOCK"].isna(),
    "Missing shot clock with >24 s in period": shots["SHOT_CLOCK"].isna() & period_seconds.gt(24),
})
pd.DataFrame({
    "attempts": flags.sum(),
    "share_pct": flags.mean().mul(100).round(2),
})

,attempts,share_pct
Missing closest-defender distance,0,0.00
Negative closest-defender distance,0,0.00
Zero closest-defender distance,835,0.82
Closest-defender distance above 25 ft,48,0.05
Zero distance to basket,4,0.00
Negative touch time,258,0.25
Missing shot clock,4626,4.54
Missing shot clock with >24 s in period,1779,1.74


### Inspect the flagged cases in context

Compare how often these flags appear on two-point and three-point attempts.

In [7]:
flags.groupby(shots["PTS_TYPE"]).mean().mul(100).round(2).T.rename(
    columns={2: "2-point attempts (%)", 3: "3-point attempts (%)"}
)

PTS_TYPE,2-point attempts (%),3-point attempts (%)
Missing closest-defender distance,0.00,0.00
Negative closest-defender distance,0.00,0.00
Zero closest-defender distance,1.08,0.09
Closest-defender distance above 25 ft,0.06,0.01
Zero distance to basket,0.01,0.00
Negative touch time,0.29,0.15
Missing shot clock,3.69,6.90
Missing shot clock with >24 s in period,1.73,1.78


 Inspect the extreme defender distances cases alongside shot distance and game clock for manual searching for that exact bucket. The checking will be done inside [Defender Distance Outliers](../docs/defender-distance-outliers.md), all exact shoots found look plausible.

In [8]:
shots.loc[flags["Closest-defender distance above 25 ft"], [
    "MATCHUP", "player_name", "PERIOD", "GAME_CLOCK", "SHOT_CLOCK",
    "PTS_TYPE", "SHOT_DIST", "CLOSE_DEF_DIST","SHOT_RESULT"
]]

,MATCHUP,player_name,PERIOD,GAME_CLOCK,SHOT_CLOCK,PTS_TYPE,SHOT_DIST,CLOSE_DEF_DIST,SHOT_RESULT
2278,"JAN 24, 2015 - CHA vs. NYK",gerald henderson,3,10:04,24.0,2,2.1,29.9,made
4877,"JAN 24, 2015 - UTA vs. BKN",gordon hayward,4,6:20,21.6,2,2.4,52.6,made
15383,"JAN 17, 2015 - GSW @ HOU",klay thompson,1,11:04,20.7,2,2.1,40.1,made
17604,"DEC 27, 2014 - GSW vs. MIN",draymond green,3,3:37,21.0,2,2.6,32.5,made
21524,"JAN 17, 2015 - ATL @ CHI",jeff teague,3,6:02,21.2,2,3.6,37.9,made
21628,"JAN 02, 2015 - ATL @ UTA",jeff teague,3,9:31,22.4,2,4.0,32.3,made
30435,"NOV 17, 2014 - NOP @ POR",anthony davis,1,8:43,21.0,2,3.5,32.4,made
31146,"DEC 31, 2014 - NOP @ SAS",jimmer dredette,5,2:36,21.1,2,3.2,34.3,made
33593,"NOV 21, 2014 - TOR vs. MIL",greivis vasquez,2,6:01,21.0,2,3.1,30.3,made
38234,"JAN 14, 2015 - DAL @ DEN",al farouq aminu,4,8:04,21.2,2,3.6,27.1,made


Same with 0 distance to basket. In this case we will perform manual checking with the help of https://www.realapp.com/ and nba play-by-play.

In [9]:
shots.loc[flags["Zero distance to basket"], [
    "MATCHUP", "player_name", "PERIOD", "GAME_CLOCK", "SHOT_CLOCK",
    "PTS_TYPE", "SHOT_DIST", "CLOSE_DEF_DIST", "SHOT_RESULT"
]]

,MATCHUP,player_name,PERIOD,GAME_CLOCK,SHOT_CLOCK,PTS_TYPE,SHOT_DIST,CLOSE_DEF_DIST,SHOT_RESULT
27349,"JAN 16, 2015 - NOP @ PHI",omer asik,3,10:21,15.2,2,0.0,5.3,made
62013,"DEC 17, 2014 - DEN vs. HOU",wilson chandler,1,6:07,11.1,2,0.0,7.6,made
74027,"JAN 26, 2015 - PHI @ NOP",luc mbah a moute,4,11:44,8.3,2,0.0,1.7,missed
94264,"JAN 30, 2015 - MIA vs. DAL",mario chalmers,2,4:28,10.9,2,0.0,3.0,missed


### Apply the four reviewed shot-distance corrections

The three close layups were by Ömer Aşık, Wilson Chandler, and Mario Chalmers. Use **2.0 ft** as a shared estimate. For reference, I pulled the NBA's 2014–15 shot charts for those three players and selected the 303 pre-February 1 shots labeled as layups with a positive recorded whole-foot distance under 5 ft. The distance calculated from the court coordinates averaged **1.95 ft**. Rounded to the nearest foot, that gives 2 ft for these three manual corrections. This is a sample for those shooters and the stated distance range.

Real App showed **7 ft** for Luc Richard Mbah a Moute's attempt, while the official NBA play-by-play records a **2-ft driving reverse layup** (event 400). We use the official NBA value of **2.0 ft** for the correction.

The next cell makes a separate dataframe called shots_reviewed for downstream work in this notebook. The raw CSV and the original shots dataframe remain available for comparison. [NBA Stats shot charts](https://www.nba.com/stats/overview) · [76ers at Pelicans play-by-play](https://www.nba.com/game/phi-vs-nop-0021400671/play-by-play)

In [10]:
corrections = pd.DataFrame(
    [
        ("21400589", 201600, 5, 2.0),  # Ömer Aşık
        ("21400378", 201163, 6, 2.0),  # Wilson Chandler
        ("21400671", 201601, 10, 2.0),  # Luc Richard Mbah a Moute
        ("21400702", 201596, 5, 2.0),  # Mario Chalmers
    ],
    columns=[*shot_key, "REVIEWED_SHOT_DIST"],
).astype({"GAME_ID": "string"})

shots_reviewed = shots.assign(SHOT_DIST_ORIGINAL=shots["SHOT_DIST"]).merge(
    corrections, on=shot_key, how="left", validate="one_to_one",
)
corrected = shots_reviewed["REVIEWED_SHOT_DIST"].notna()
if corrected.sum() != len(corrections) or not shots_reviewed.loc[corrected, "SHOT_DIST_ORIGINAL"].eq(0).all():
    raise ValueError("Expected each reviewed shot to match one zero-distance source row.")

shots_reviewed.loc[corrected, "SHOT_DIST"] = shots_reviewed.loc[corrected, "REVIEWED_SHOT_DIST"]
shots_reviewed = shots_reviewed.drop(columns="REVIEWED_SHOT_DIST")
shots_reviewed.loc[corrected, [
    *shot_key, "player_name", "SHOT_DIST_ORIGINAL", "SHOT_DIST",
]]

,GAME_ID,player_id,SHOT_NUMBER,player_name,SHOT_DIST_ORIGINAL,SHOT_DIST
21557,21400589,201600,5,omer asik,0.0,2.0
49396,21400378,201163,6,wilson chandler,0.0,2.0
58914,21400671,201601,10,luc mbah a moute,0.0,2.0
75014,21400702,201596,5,mario chalmers,0.0,2.0


## Proposed handling for the remaining flags

| Flag | Pre-February 1 count | Proposed treatment |
| --- | ---: | --- |
| Closest defender above 25 ft | 48 | Keep the recorded distances. The clips reviewed looked plausible. Rows without footage are plausible but unverified. |
| Closest defender exactly 0 ft | 835 | Keep the raw values and compare models with zeros preserved versus treated as unknown. There are 835 zeros and 468 values at 0.1 ft, so the exact-zero concentration deserves a sensitivity check. |
| Negative touch time | 258 | Treat as missing if touch time is used, and preserve a flag for the source anomaly. The smallest value is −163.6 seconds. |
| Missing shot clock | 4,626 | Keep as missing. Of these, 2,847 have less than 24 seconds left in the period, that could be solve with just equalizing the period with shot clock on this cases. Test whether adding shot clock helps a model that already has game clock. |

These proposals keep the attempts available for training. Compare treatments using earlier games for fitting and later games for validation within the EDA window. Use the February holdout for final evaluation.